# OpenFOAM CFD Setup — Toroidal Propeller Simulation

This notebook:
1. Installs OpenFOAM v2212 on Google Colab (Ubuntu)
2. Generates propeller STL geometry
3. Creates OpenFOAM case directories
4. Runs a representative case (medium-gap toroidal, 4000 RPM, static thrust)
5. Post-processes forces and compares with BEMT / experimental data

> **Note:** OpenFOAM installation takes ~5–8 minutes on Colab. A full propeller
> simulation requires ~30–120 minutes depending on mesh refinement and hardware.
> For faster results, use the BEMT notebook (`01_BEMT_Analysis.ipynb`).

---
**Solver:** simpleFoam (steady-state RANS)  
**Turbulence:** k-ω SST  
**Rotation:** Multiple Reference Frame (MRF)

In [ ]:
# ── Step 1: Clone repository ───────────────────────────────────────────
import os, sys

REPO_URL = 'https://github.com/rp3gregorio/Propeller-CFD.git'
REPO_DIR = '/content/Propeller-CFD'
BRANCH   = 'claude/add-cfd-propeller-simulation-EEQqr'

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Repo directory:', REPO_DIR)

In [ ]:
# ── Step 2: Install OpenFOAM v2212 (ESI) on Ubuntu/Colab ──────────────
# This takes ~5-8 minutes on first run.
%%bash
set -e

if [ -f /usr/bin/blockMesh ]; then
    echo 'OpenFOAM already installed.'
    exit 0
fi

echo '--- Adding OpenFOAM apt repository ---'
curl -s https://dl.openfoam.com/add-apt-repository.sh | bash
apt-get update -qq
apt-get install -y openfoam2212-default 2>&1 | tail -5

echo '--- Sourcing OpenFOAM environment ---'
echo 'source /usr/lib/openfoam/openfoam2212/etc/bashrc' >> /root/.bashrc
source /usr/lib/openfoam/openfoam2212/etc/bashrc
echo "OpenFOAM version: $(foamVersion)"
echo 'Installation complete.'

In [ ]:
# ── Step 3: Set OpenFOAM environment for Python subprocess calls ────────
import subprocess

OF_ENV_CMD = 'source /usr/lib/openfoam/openfoam2212/etc/bashrc && '

def run_of(cmd, cwd=None):
    """Run an OpenFOAM command with the correct environment."""
    full_cmd = f'bash -c "{OF_ENV_CMD}{cmd}"'
    result = subprocess.run(full_cmd, shell=True, cwd=cwd,
                            capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-2000:] if result.stderr else '(none)')
        raise RuntimeError(f'Command failed: {cmd}')
    return result.stdout

print(run_of('foamVersion'))

In [ ]:
# ── Step 4: Generate propeller STL geometry ─────────────────────────────
!pip install -q numpy
!python geometry/generate_stl.py

import os
stl_files = os.listdir('geometry/stl')
print('Generated STL files:', stl_files)

In [ ]:
# ── Step 5: Create a representative OpenFOAM case ──────────────────────
# Running: toroidal_medium_gap at 4000 RPM, static thrust
# (Change these parameters to run different conditions)

CONFIG    = 'toroidal_medium_gap'   # or: conventional, toroidal_low_gap, toroidal_high_gap
CASE_TYPE = 'static_thrust'          # or: wind_tunnel
RPM       = 4000                     # 1000 – 6000
V_INF     = 0.0                      # 0.0 / 3.0 / 9.0 / 15.0 m/s
N_PROCS   = 2                        # Colab has 2 CPUs

!python openfoam/generate_case.py \
    --config {CONFIG} \
    --case_type {CASE_TYPE} \
    --rpm {RPM} \
    --v_inf {V_INF} \
    --n_procs {N_PROCS}

CASE_DIR = f'runs/{CONFIG}_{RPM}rpm_{CASE_TYPE}'
print('Case directory:', CASE_DIR)
!ls {CASE_DIR}

In [ ]:
# ── Step 6: Run blockMesh ──────────────────────────────────────────────
print('Running blockMesh...')
out = run_of('blockMesh', cwd=CASE_DIR)
print(out[-500:])
print('blockMesh done.')

In [ ]:
# ── Step 7: Run snappyHexMesh ─────────────────────────────────────────
print('Running snappyHexMesh (may take a few minutes)...')
out = run_of('snappyHexMesh -overwrite', cwd=CASE_DIR)
print(out[-800:])
print('snappyHexMesh done.')

In [ ]:
# ── Step 8: Check mesh quality ────────────────────────────────────────
out = run_of('checkMesh', cwd=CASE_DIR)
# Extract key mesh quality metrics
for line in out.split('\n'):
    if any(k in line for k in ['cells:', 'Max skewness', 'Max non-ortho', 'Overall']):
        print(line.strip())

In [ ]:
# ── Step 9: Run simpleFoam solver ─────────────────────────────────────
import shutil

# Copy initial conditions
orig = os.path.join(CASE_DIR, '0.orig')
zero = os.path.join(CASE_DIR, '0')
if os.path.isdir(orig) and not os.path.isdir(zero):
    shutil.copytree(orig, zero)

print('Running simpleFoam...')
if N_PROCS > 1:
    run_of('decomposePar', cwd=CASE_DIR)
    run_of(f'mpirun -np {N_PROCS} simpleFoam -parallel', cwd=CASE_DIR)
    run_of('reconstructPar', cwd=CASE_DIR)
else:
    run_of('simpleFoam', cwd=CASE_DIR)
print('simpleFoam done.')

In [ ]:
# ── Step 10: Extract and plot forces ──────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import re

force_glob = os.path.join(CASE_DIR, 'postProcessing', 'propellerForces')

def read_forces_notebook(case_dir):
    pb = os.path.join(case_dir, 'postProcessing', 'propellerForces')
    if not os.path.isdir(pb):
        return None
    time_dirs = sorted([d for d in os.listdir(pb) if os.path.isdir(os.path.join(pb, d))],
                        key=lambda x: float(x) if x.replace('.','').isdigit() else 0)
    if not time_dirs:
        return None
    ff = os.path.join(pb, time_dirs[-1], 'force.dat')
    if not os.path.isfile(ff):
        return None
    data = []
    with open(ff) as f:
        for line in f:
            if line.startswith('#') or not line.strip(): continue
            nums = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', line)
            if len(nums) >= 13:
                data.append([float(x) for x in nums[:13]])
    if not data: return None
    arr = np.array(data)
    return arr[:, 0], arr[:, 3] + arr[:, 6], arr[:, 11] + arr[:, 12]

result = read_forces_notebook(CASE_DIR)

if result is not None:
    t_iters, Fz, Mz = result
    T_converged = Fz[-50:].mean() / 9.81 * 1000
    Q_converged = Mz[-50:].mean()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(t_iters, Fz / 9.81 * 1000)
    ax1.axhline(T_converged, color='red', ls='--', label=f'Converged: {T_converged:.1f} gf')
    ax1.set_xlabel('Iteration'); ax1.set_ylabel('Thrust [gf]')
    ax1.set_title('Thrust Convergence'); ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(t_iters, Mz * 1000)
    ax2.axhline(Q_converged * 1000, color='red', ls='--', label=f'Converged: {Q_converged*1000:.2f} mN·m')
    ax2.set_xlabel('Iteration'); ax2.set_ylabel('Torque [mN·m]')
    ax2.set_title('Torque Convergence'); ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/plots/cfd_convergence.png', dpi=150)
    plt.show()

    print(f'\n=== CFD Result ({CONFIG}, {RPM} RPM) ===')
    print(f'  Thrust : {T_converged:.1f} gf = {T_converged*9.81/1000:.3f} N')
    print(f'  Torque : {Q_converged*1000:.2f} mN·m')

    # Compare with BEMT
    from bemt.bemt_solver import BEMTSolver
    from bemt.propeller_configs import CONFIGS
    solver = BEMTSolver(CONFIGS[CONFIG], n_sections=60)
    bemt_r = solver.solve(RPM, V_INF)
    print(f'  BEMT T : {bemt_r["T"]/9.81*1000:.1f} gf')
    print(f'  Diff   : {(T_converged - bemt_r["T"]/9.81*1000)/bemt_r["T"]*9.81*1000*100:+.1f}%  (CFD vs BEMT)')
else:
    print('Force data not found — simulation may not have completed yet.')
    print('Run Allrun script manually or check log files.')

## Next Steps

### Run all cases
```bash
# Generate all 48 cases (4 configs × 3 RPMs × 4 wind speeds)
bash scripts/setup_all_cases.sh

# Run all cases (sequential on local machine)
bash scripts/run_all_cases.sh

# Post-process all cases
python postprocessing/extract_forces.py --runs_dir runs/

# Generate comparison plots (CFD + BEMT + Experimental)
python postprocessing/plot_comparison.py --source both --runs_dir runs/
```

### Replace parametric geometry with actual CAD
1. Export your propeller CAD as STL (units: metres)
2. Copy to `geometry/stl/<config_name>.stl`
3. Re-run the case setup and simulation

### Increase mesh resolution
Edit `openfoam/templates/*/system/snappyHexMeshDict`:
- Increase `level` values in `refinementSurfaces` (e.g., `(6 8)` for finer near-wall mesh)
- Increase background grid in `blockMeshDict` (e.g., `(30 30 40)` for denser background)

### Transient simulation (sliding mesh)
Replace `simpleFoam` with `pimpleFoam` and change MRF to AMI (Arbitrary Mesh Interface) for higher-fidelity time-resolved predictions.